# Step 05 — Ultra-fast, streaming HMM estimation panel (v2)

**Output:** one trip-level Parquet file, with 38 variables. Annual events/demographics are repeated within each household-year. The latent state still evolves **by year**, not by trip.

**Performance design:** read the demographic TSV once, join one annual trip partition at a time, stream Arrow batches directly into **one** ParquetWriter, and validate while writing. No global sort, 37M-row `COUNT(DISTINCT)`, full-output DuckDB scan, or intermediate 10-year dataset.

**Missing transition covariates:** Replace missing values in the seven annual life-event indicators with 0 in the estimation panel. This is a modeling convention; the original annual dataset remains unchanged.

**Safety:** never overwrite a valid output until the whole new file is complete. If interrupted, discard the `.partial.parquet` file; the final output remains unchanged. All restricted data and audits stay outside GitHub.

**Requirements:** `duckdb`, `pyarrow`, and `pandas` are installed in the active VS Code kernel. Run cells in order. This notebook is optimized for throughput, not for a particular duration; storage throughput can still dominate runtime.\n**Age validation:** Invalid head ages computed from birth years are replaced with their Nielsen age-band midpoints (open bins use 22 and 70), and fallback counts are printed. This affects only suspect birth-derived ages; valid ages remain unchanged.\n

In [ ]:
from pathlib import Path
from time import perf_counter
import gc
import duckdb
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
import pandas as pd

DATA_ROOT = Path(r"C:/Users/rwen13/Documents/healthy_purchase_data")
MASTER = DATA_ROOT / "processed/hmm_panel/hmm_master"
TRIP_DIR = MASTER / "hmm_trips"
ANNUAL_FILE = MASTER / "hmm_annual_covariates_income_imputed.parquet"
DEMO_FILE = DATA_ROOT / "processed/demographics/household_demographics_with_events_2008_2017.tsv"
INITIAL_FILE = MASTER / "hmm_initial_fresh_fv_share_2007.parquet"
OUTPUT_DIR = DATA_ROOT / "processed/hmm_input"
OUTPUT_FILE = OUTPUT_DIR / "hmm_estimation_panel.parquet"
PARTIAL_FILE = OUTPUT_DIR / "hmm_estimation_panel.partial.parquet"
AUDIT_DIR = OUTPUT_DIR / "validation"

# Set to True only when you explicitly intend to replace an existing final file.
OVERWRITE_EXISTING = False
MEMORY_LIMIT = "4GB"
THREADS = 4
BATCH_ROWS = 250_000  # Streaming batches; decrease to 100_000 if memory is tight.
COMPRESSION = "snappy"  # Faster to write than ZSTD; larger output.

EMISSIONS = ["fv_y", "milk_y", "yogurt_y", "cereal_y", "snack_y", "beverage_y"]
EVENTS = ["event_new_child", "getting_married", "getting_divorce_or_separation",
          "moving", "getting_job", "retirement", "other_unemployment"]
SPENDING = ["fresh_spending", "non_fresh_spending", "skimmed_milk_spending",
            "low_fat_milk_spending", "other_milk_spending", "skimmed_yogurt_spending",
            "low_fat_yogurt_spending", "other_yogurt_spending",
            "low_sugar_cereal_spending", "medium_sugar_cereal_spending", "high_sugar_cereal_spending",
            "healthy_snack_spending", "unhealthy_snack_spending",
            "healthy_beverage_spending", "unhealthy_beverage_spending"]

for p in [ANNUAL_FILE, DEMO_FILE, INITIAL_FILE]:
    if not p.is_file():
        raise FileNotFoundError(f"Missing required input: {p}")
trip_files = sorted(TRIP_DIR.glob("panel_year=*/trips.parquet"))
if not trip_files:
    raise FileNotFoundError(f"No trip partitions found at {TRIP_DIR}")
trip_by_year = {int(p.parent.name.split("=", 1)[1]): p for p in trip_files}
if set(trip_by_year) != set(range(2008, 2018)) or len(trip_files) != 10:
    raise ValueError(f"Expected exactly 2008–2017 trip files; got {sorted(trip_by_year)}")
if OUTPUT_FILE.exists() and not OVERWRITE_EXISTING:
    raise FileExistsError(f"Final output exists: {OUTPUT_FILE}. Set OVERWRITE_EXISTING=True to replace it.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
(DATA_ROOT / "temp").mkdir(parents=True, exist_ok=True)

# Never remove the final output before a replacement is fully created.
if PARTIAL_FILE.exists():
    print(f"Deleting leftover INCOMPLETE temporary file: {PARTIAL_FILE}", flush=True)
    PARTIAL_FILE.unlink()

def qpath(path):
    return "'" + str(path).replace(chr(92), "/").replace("'", "''") + "'"

con = duckdb.connect(database=":memory:")
con.execute(f"SET memory_limit = '{MEMORY_LIMIT}'")
con.execute(f"SET threads = {THREADS}")
con.execute(f"SET temp_directory = {qpath(DATA_ROOT / 'temp')}")
print(f"Ready: {len(trip_files)} yearly input files; {BATCH_ROWS:,} rows per streaming batch.", flush=True)

## 1. Prepare small lookup tables once

The demographic TSV is parsed only once with explicit tab/quote settings (fixes the earlier DuckDB parser issue). The resulting lookup has one record per `(hhid, panel_year)`. Validate uniqueness and age/income coverage **before** reading the large trip partitions.

In [ ]:
t0 = perf_counter()
# The data have many columns: never supply a truncated four-column CSV schema.
# Validate each head's birth-derived age before taking the household-head mean.
# Out-of-range birth-derived ages are replaced with the corresponding Nielsen age-band midpoint.
# Open-ended age bands (<25 and 65+) use documented assumptions of 22 and 70 years.
# Missing/absent heads are not treated as age zero.
con.execute(f"""
    CREATE TEMP TABLE demo_age AS
    WITH raw AS (
        SELECT CAST(hhid AS VARCHAR) AS hhid,
               TRY_CAST(panel_year AS INTEGER) AS panel_year,
               TRY_CAST(male_head_age_from_birth AS DOUBLE) AS male_birth_age,
               TRY_CAST(female_head_age_from_birth AS DOUBLE) AS female_birth_age,
               TRY_CAST(Male_Head_Age AS INTEGER) AS male_age_code,
               TRY_CAST(Female_Head_Age AS INTEGER) AS female_age_code
        FROM read_csv({qpath(DEMO_FILE)}, delim='\t', header=true,
                      all_varchar=true, quote='', escape='', strict_mode=true)
    ), coded AS (
        SELECT *,
            CASE male_age_code
                WHEN 1 THEN 22.0 WHEN 2 THEN 27.0 WHEN 3 THEN 32.0
                WHEN 4 THEN 37.0 WHEN 5 THEN 42.0 WHEN 6 THEN 47.0
                WHEN 7 THEN 52.0 WHEN 8 THEN 59.5 WHEN 9 THEN 70.0
            END AS male_band_age,
            CASE female_age_code
                WHEN 1 THEN 22.0 WHEN 2 THEN 27.0 WHEN 3 THEN 32.0
                WHEN 4 THEN 37.0 WHEN 5 THEN 42.0 WHEN 6 THEN 47.0
                WHEN 7 THEN 52.0 WHEN 8 THEN 59.5 WHEN 9 THEN 70.0
            END AS female_band_age
        FROM raw
    ), selected AS (
        SELECT *,
               CASE WHEN male_birth_age BETWEEN 18 AND 110 THEN male_birth_age
                    ELSE male_band_age END AS male_age,
               CASE WHEN female_birth_age BETWEEN 18 AND 110 THEN female_birth_age
                    ELSE female_band_age END AS female_age,
               (male_birth_age IS NOT NULL AND NOT (male_birth_age BETWEEN 18 AND 110))
                   AS male_birth_age_invalid,
               (female_birth_age IS NOT NULL AND NOT (female_birth_age BETWEEN 18 AND 110))
                   AS female_birth_age_invalid
        FROM coded
    )
    SELECT hhid, panel_year,
           CASE WHEN male_age IS NOT NULL AND female_age IS NOT NULL
                    THEN (male_age + female_age) / 2.0
                ELSE COALESCE(male_age, female_age) END AS age,
           male_birth_age_invalid, female_birth_age_invalid,
           CASE WHEN male_birth_age_invalid AND male_band_age IS NOT NULL THEN 1 ELSE 0 END
               + CASE WHEN female_birth_age_invalid AND female_band_age IS NOT NULL THEN 1 ELSE 0 END
               AS age_head_fallback_count
    FROM selected
""")
con.execute(f"""
    CREATE TEMP TABLE annual_keyed AS
    SELECT CAST(hhid AS VARCHAR) AS hhid,
           CAST(panel_year AS INTEGER) AS panel_year,
-- Modeling convention: missing annual transition event indicators are treated as no event (0).
           {', '.join('COALESCE(TRY_CAST(' + v + ' AS TINYINT), 0)::TINYINT AS ' + v for v in EVENTS)},
           TRY_CAST(K_1 AS TINYINT) AS K_1,
           TRY_CAST(income10k_imputed AS DOUBLE) AS income10k_imputed,
           TRY_CAST(family_size_numeric AS DOUBLE) AS family_size_numeric
    FROM read_parquet({qpath(ANNUAL_FILE)})
""")

for name in ("annual_keyed", "demo_age"):
    duplicate = con.execute(f"""
        SELECT hhid, panel_year, COUNT(*) AS n
        FROM {name} GROUP BY hhid, panel_year HAVING COUNT(*) > 1 LIMIT 1
    """).fetchone()
    if duplicate:
        raise ValueError(f"Duplicate household-year key in {name}: {duplicate}")

coverage = con.execute("""
    SELECT COUNT(*) AS annual_rows,
           COUNT(d.hhid) AS matched_age_rows,
           COUNT(*) FILTER (WHERE d.age IS NULL) AS missing_age,
           COUNT(*) FILTER (WHERE d.age NOT BETWEEN 18 AND 110) AS invalid_age,
           COUNT(*) FILTER (WHERE a.income10k_imputed IS NULL) AS missing_income
    FROM annual_keyed a LEFT JOIN demo_age d USING (hhid, panel_year)
""").fetchone()
print("Annual rows | age matches | missing age | invalid age | missing income:", coverage, flush=True)
# Lightweight age QA: report source-age anomalies and category-based corrections.
age_fallback_audit = con.execute("""
    SELECT COUNT(*) FILTER (WHERE male_birth_age_invalid OR female_birth_age_invalid)
               AS household_years_with_invalid_head_birth_age,
           COALESCE(SUM(age_head_fallback_count), 0) AS corrected_head_age_values,
           COUNT(*) FILTER (WHERE age IS NULL OR age NOT BETWEEN 18 AND 110)
               AS unresolved_household_years
    FROM demo_age
""").fetchone()
print("Age QA | anomalous HH-years | head values corrected | unresolved:", age_fallback_audit, flush=True)

if coverage[0] != coverage[1] or any(coverage[i] for i in (2, 3, 4)):
    raise ValueError("Annual covariates fail age or income coverage checks.")

pi_values = con.execute(f"""
    SELECT TRY_CAST(initial_fresh_fv_share_2007 AS DOUBLE)
    FROM read_parquet({qpath(INITIAL_FILE)})
""").fetchall()
if len(pi_values) != 1 or pi_values[0][0] is None or not (0 < pi_values[0][0] < 1):
    raise ValueError(f"Expected exactly one common initialization share in (0, 1): {pi_values}")
PI = float(pi_values[0][0])

# Pre-check source columns without scanning any trip rows.
required_trip = set(["hhid", "panel_year", "trip_id", "purchase_date", *EMISSIONS, *SPENDING, "food_spend"])
expected_rows_by_year = {}
for year, trip_file in trip_by_year.items():
    pf = pq.ParquetFile(trip_file)
    missing = required_trip - set(pf.schema_arrow.names)
    if missing:
        raise ValueError(f"Missing trip columns for {year}: {sorted(missing)}")
    expected_rows_by_year[year] = pf.metadata.num_rows
print(f"Lookup validated in {perf_counter()-t0:.1f}s; common pi start={PI:.8f}", flush=True)
print(f"Source trip rows: {sum(expected_rows_by_year.values()):,}", flush=True)

## 2. Single-pass streaming merge and Parquet write

Each year is joined **one at a time** and streamed directly to one ParquetWriter. Progress is printed after each batch and year. Basic value-range and missingness checks are performed on each batch already in memory—no expensive second read of the output.

**Important:** this step assumes Step 03 already validated unique trip keys; Step 05 verifies each year's row count is unchanged by joins. A source with duplicated trip IDs can still require upstream repair.

In [ ]:
selected_emissions = ", ".join(f"TRY_CAST(t.{v} AS TINYINT) AS {v}" for v in EMISSIONS)
selected_events = ", ".join(f"a.{v} AS {v}" for v in EVENTS)
selected_spending = ", ".join(f"TRY_CAST(t.{v} AS DOUBLE) AS {v}" for v in SPENDING)

# Accumulate small audit counters during the same streaming pass.
rows_written = 0
rows_by_year = []
missing_events = {v: 0 for v in EVENTS}  # Post-fill NULL audit: must remain zero.
bad_emissions = {v: 0 for v in EMISSIONS}
missing_food_spend = 0
writer = None
started = perf_counter()

try:
    for year in sorted(trip_by_year):
        file = trip_by_year[year]
        expected = expected_rows_by_year[year]
        year_start = perf_counter()
        year_rows = 0
        print(f"[{year}] START | source rows={expected:,}", flush=True)

        # No ORDER BY: trip order is not part of the HMM data contract.
        sql = f"""
            SELECT CAST(t.hhid AS VARCHAR) AS hhid,
                   CAST(t.panel_year AS INTEGER) AS panel_year,
                   CAST(t.trip_id AS VARCHAR) AS trip_id,
                   CAST(t.purchase_date AS DATE) AS purchase_date,
                   {selected_emissions}, {selected_events},
                   a.K_1 AS K_1,
                   a.income10k_imputed AS income10k_imputed,
                   a.family_size_numeric AS family_size_numeric,
                   d.age AS age,
                   {selected_spending},
                   TRY_CAST(t.food_spend AS DOUBLE) AS food_spend,
                   {PI:.17g}::DOUBLE AS initial_fresh_fv_share_2007
            FROM read_parquet({qpath(file)}) t
            INNER JOIN annual_keyed a
              ON CAST(t.hhid AS VARCHAR) = a.hhid
             AND CAST(t.panel_year AS INTEGER) = a.panel_year
            INNER JOIN demo_age d
              ON a.hhid = d.hhid AND a.panel_year = d.panel_year
        """
        # DuckDB streams record batches; nothing materializes 37M trip rows in RAM.
        reader = con.execute(sql).fetch_record_batch(rows_per_batch=BATCH_ROWS)
        for batch_no, batch in enumerate(reader, start=1):
            if writer is None:
                writer = pq.ParquetWriter(
                    str(PARTIAL_FILE), batch.schema, compression=COMPRESSION,
                    use_dictionary=True, write_statistics=True
                )
            if not batch.schema.equals(writer.schema, check_metadata=False):
                raise ValueError(f"Inconsistent output schema detected for year {year}")

            # Lightweight batch-level validation, while data are already loaded.
            for name in EMISSIONS:
                arr = batch.column(batch.schema.get_field_index(name))
                bad_emissions[name] += arr.null_count
                bounds = pc.min_max(arr).as_py()
                if bounds['min'] is not None and bounds['min'] < 0:
                    raise ValueError(f"Invalid negative {name} in year {year}")
                if bounds['max'] is not None and bounds['max'] > (3 if name == 'cereal_y' else 2):
                    raise ValueError(f"Invalid {name} > allowed range in year {year}")
            for name in EVENTS:
                arr = batch.column(batch.schema.get_field_index(name))
                missing_events[name] += arr.null_count
                if arr.null_count:
                    raise ValueError(f"Missing {name} remained after zero-filling in year {year}")
            for name in ("age", "income10k_imputed"):
                arr = batch.column(batch.schema.get_field_index(name))
                if arr.null_count:
                    raise ValueError(f"Missing {name} in {year}: {arr.null_count} trip rows")
            amount = batch.column(batch.schema.get_field_index("food_spend"))
            missing_food_spend += amount.null_count  # Preserve NA; never silently fill with zero.
            minimum = pc.min(amount).as_py()
            if minimum is not None and minimum < 0:
                raise ValueError(f"Negative food_spend detected in {year}")
            if batch.column(batch.schema.get_field_index("panel_year")).null_count:
                raise ValueError(f"Missing panel_year in {year}")

            writer.write_table(pa.Table.from_batches([batch]), row_group_size=BATCH_ROWS)
            year_rows += batch.num_rows
            rows_written += batch.num_rows
            if batch_no % 5 == 0:
                print(f"[{year}] {year_rows:,}/{expected:,} trip rows written", flush=True)

        if year_rows != expected:
            raise ValueError(
                f"Year {year}: expected {expected:,} source trips but wrote {year_rows:,}. "
                "Annual/demographic join dropped or duplicated trips."
            )
        rows_by_year.append({"panel_year": year, "source_trips": expected,
                             "output_trips": year_rows, "elapsed_seconds": round(perf_counter()-year_start, 2)})
        print(f"[{year}] COMPLETE | {year_rows:,} trips | {perf_counter()-year_start:.1f}s", flush=True)
        gc.collect()

finally:
    if writer is not None:
        writer.close()  # On failure, the .partial file is NOT published.

if rows_written != sum(expected_rows_by_year.values()):
    raise ValueError("Total source/output trip row counts differ.")
if any(bad_emissions.values()):
    raise ValueError(f"Missing or invalid emission outcomes: {bad_emissions}")
print(f"STREAMING BUILD COMPLETE: {rows_written:,} trips, {perf_counter()-started:.1f}s", flush=True)

## 3. Instant metadata check, small audits, and atomic publication

No full-output SQL scan or global distinct count. Reported `rows_written` comes from streamed batches, and the Parquet footer independently records the output row count. The final file is published only after this comparison succeeds.

In [ ]:
metadata = pq.read_metadata(PARTIAL_FILE)
expected_total = sum(expected_rows_by_year.values())
if metadata.num_rows != expected_total or metadata.num_rows != rows_written:
    raise ValueError(f"Parquet footer row count mismatch: {metadata.num_rows:,} vs {expected_total:,}")
if metadata.num_columns != 38:
    raise ValueError(f"Expected 38 variables, found {metadata.num_columns}")

# These are small, non-sensitive aggregate checks, kept outside the repository.
pd.DataFrame(rows_by_year).to_csv(AUDIT_DIR / "step05_streaming_rows_by_year.csv", index=False)
pd.DataFrame([{"variable": v, "missing_trip_rows": missing_events[v]} for v in EVENTS]).to_csv(
    AUDIT_DIR / "step05_streaming_missing_life_events.csv", index=False
)
pd.DataFrame([{"variable": v, "missing_or_invalid": bad_emissions[v]} for v in EMISSIONS] +
             [{"variable": "food_spend", "missing_or_invalid": missing_food_spend}]).to_csv(
    AUDIT_DIR / "step05_streaming_value_audit.csv", index=False
)

# Replace only after the new Parquet is closed and validated.
PARTIAL_FILE.replace(OUTPUT_FILE)
print(f"SAVED: {OUTPUT_FILE}", flush=True)
print(f"ROWS: {metadata.num_rows:,} | COLUMNS: {metadata.num_columns} | SIZE: {OUTPUT_FILE.stat().st_size/1024**3:.2f} GiB", flush=True)
print(f"Missing food_spend (retained as NULL): {missing_food_spend:,}", flush=True)
print("Null life-event trip counts:", missing_events, flush=True)
con.close()

### Notes for model estimation

- This is a **single file**, as requested; repeated covariates do not imply trip-level state transitions.
- `hhid` and `trip_id` remain strings. Emission outcomes are small integers; category spending is numeric.
- Missing life-event / transition covariate values are set to `0` in the estimation panel (interpreted as no recorded event); the original annual input is not modified. This is an explicit modeling assumption, not evidence that the event did not occur.
- The common 2007 fresh-FV share is an **initialization value** for the HMM's common Healthy-state probability, not a fixed latent-state probability.
- Because the output is not globally sorted, **sort internally by household/year/trip when building HMM arrays**. Handle calendar-year gaps explicitly.
- `food_spend` may be missing for some unmatched trips; it is not filled with zero.
- A row count comparison verifies no trips were lost or multiplied. It does **not** prove trip-key uniqueness or purchase-level Magnet monetary reconciliation; those remain Step 03/02 responsibilities.